In [1]:
import sys
sys.path.append("..")

from pu.data.loaders import FullCSVLoader
from pu.data.pu_builder import build_pu_data
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
import torch.backends.cudnn as cudnn
import numpy as np
import torchvision
from torchvision import datasets, models, transforms
from torchvision.io import read_image

import timm
from timm.data import resolve_data_config
from timm.data.transforms_factory import create_transform

from tqdm import tqdm

import peft

import matplotlib.pyplot as plt
import time
import os
from PIL import Image
from tempfile import TemporaryDirectory

In [2]:
def get_all_paths():

    dataset_params = {
        'ava': ['/srv/PU-dataset/unlabeled.csv', 'id', '/srv/PU-dataset/dataset_unlabeled', None],
        'aadb_train': ['/srv/aadb/train.csv', 'path', '/srv/aadb', None],
        'aadb_val': ['/srv/aadb/validation.csv', 'path', '/srv/aadb', None],
        'aadb_test': ['/srv/aadb/testnew.csv', 'path', '/srv/aadb', None],
        'laion_aes': ['/srv/PU-dataset/positive.csv', 'path', '/srv/PU-dataset/dataset_positive', None],
        'cima': ['/srv/PU-dataset/cima.csv', 'path', '/srv/cima', 'ඞ']
    }

    all_paths = {}

    for dataset in dataset_params:
        featureset_name = f"{dataset}"
        loader = FullCSVLoader(*dataset_params[dataset][:-1])

        path_col = dataset_params[dataset][1]
        data = loader.load_data(sep=dataset_params[dataset][3])

        data = data.rename(columns={path_col: 'path'})
        all_paths[featureset_name] = data

    all_paths['aadb'] = pd.concat([all_paths[f'aadb_{split}'] for split in ['train', 'val', 'test']])
    all_paths['aadb']['label'] = all_paths['aadb']['label'] * 10

    return all_paths

In [3]:
def get_ava_datasets(paths, model, quantile):

    class PUDataset(torch.utils.data.Dataset):
        def __init__(self, paths, labels):
            self.paths = paths
            self.labels = labels
            self.transform = create_transform(**resolve_data_config(model.pretrained_cfg, model=model))
    
        def __len__(self):
            return len(self.paths)
    
        def __getitem__(self, idx):
            image = read_image(self.paths[idx], mode=torchvision.io.image.ImageReadMode.RGB)
            image = torchvision.transforms.functional.convert_image_dtype(image, torch.float32)
            image = self.transform(image)
            label = self.labels[idx].astype(np.float32)
            return image,label
    
    X_train, X_val, X_test, y_train, y_val, y_test, y_test_pu = build_pu_data(
            paths['ava'],
            frac=1.0,
            move_to_unlabeled_frac=0.0,
            val_split=0.2,
            val_split_positive='same',
            reliable_positive_fn=lambda row, df: row['VotesMean'] > quantile,
            positive_fn=lambda row, df: row['VotesMean'] >= 5.0,
            test_frac=0.2,
            input_mode='images',
            random_state=1234
        )

    shuffle_order = np.random.permutation(len(X_train))
    X_train, y_train = X_train[shuffle_order], y_train[shuffle_order]
    
    paths_labels_splits = {
        'train': (X_train, y_train),
        'val': (X_val, y_val),
        'test': (X_test, y_test),
        'test_pu': (X_test, y_test_pu)
    }

    loaders = {}
    
    for split in paths_labels_splits:
        dataset = PUDataset(*paths_labels_splits[split])
        loaders[split] = torch.utils.data.DataLoader(dataset, batch_size=32, shuffle=True, num_workers=16, pin_memory=True)

    return loaders

In [4]:
def train(model, optimizer, criterion, train_dataloader, valid_dataloader, epochs):
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for batch in tqdm(train_dataloader, f"Training epoch {epoch}"):
            xb, yb = batch[0], batch[1]
            xb, yb = xb.to(device), yb.to(device)
            outputs = model(xb)
            lsm = torch.squeeze(torch.sigmoid(outputs))
            loss = criterion(lsm, yb)
            train_loss += loss.detach().float()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

        model.eval()
        valid_loss = 0
        correct = 0
        n_total = 0
        for batch in tqdm(valid_dataloader, f"Validation epoch {epoch}"):
            xb, yb = batch[0], batch[1]
            xb, yb = xb.to(device), yb.to(device)
            with torch.no_grad():
                outputs = model(xb)
            lsm = torch.squeeze(torch.sigmoid(outputs))
            loss = criterion(lsm, yb)
            valid_loss += loss.detach().float()
            correct += (torch.gt(lsm, 0.5).long() == yb).sum().item()
            n_total += len(yb)

        train_loss_total = (train_loss / len(train_dataloader)).item()
        valid_loss_total = (valid_loss / len(valid_dataloader)).item()
        valid_acc_total = correct / n_total
        print(f"{epoch=:<2}  {train_loss_total=:.4f}  {valid_loss_total=:.4f}  {valid_acc_total=:.4f}")

In [5]:
def test(model, test_dataloader):
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    model.eval()
    correct = 0
    n_total = 0
    for batch in tqdm(test_dataloader):
        xb, yb = batch[0], batch[1]
        xb, yb = xb.to(device), yb.to(device)
        with torch.no_grad():
            outputs = model(xb)
        lsm = torch.squeeze(torch.sigmoid(outputs))
        correct += (torch.gt(lsm, 0.5).long() == yb).sum().item()
        n_total += len(yb)

    test_acc_total = correct / n_total
    print(f"Testing accuracy: {test_acc_total=:.4f}")

In [6]:
class NNPULoss(nn.Module):
    def __init__(self, prior, base_loss):
        super(NNPULoss, self).__init__()
        self.prior = prior
        self.base_loss = base_loss
        self.has_found_nan = False

    def forward(self, predictions, targets):
        positive_examples = predictions[targets == 1.0]
        unlabeled_examples = predictions[targets == 0.0]

        positive_positive_risk = self.prior * torch.mean(self.base_loss(positive_examples, torch.ones_like(positive_examples)))
        unlabeled_negative_risk = torch.mean(self.base_loss(unlabeled_examples, torch.zeros_like(unlabeled_examples)))
        positive_negative_risk = self.prior * torch.mean(self.base_loss(positive_examples, torch.zeros_like(positive_examples)))

        loss = positive_positive_risk + nn.functional.relu(unlabeled_negative_risk - positive_negative_risk)
        
        return loss

In [7]:
def patch_lora(model):
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    config = peft.LoraConfig(r=8, target_modules=r"blocks\.\d*\.attn\.(qkv|proj)", modules_to_save=["head.fc"])
    return peft.get_peft_model(model, config).to(device)

In [8]:
ava_quantiles = [5.0, 5.386517, 5.475771, 5.566116, 5.660284, 5.758871, 5.865385, 5.987416, 6.129032, 6.307692, 6.574194]

paths = get_all_paths()
model = timm.create_model('vit_large_patch14_clip_224.openai', pretrained=True, num_classes=1)
loaders = get_ava_datasets(paths, model, ava_quantiles[9])
model = patch_lora(model)
optimizer = torch.optim.Adam(model.parameters(), lr=2e-4)
criterion = NNPULoss(0.7, torch.nn.BCELoss())
model.print_trainable_parameters()

trainable params: 1,179,648 || all params: 304,360,449 || trainable%: 0.3876


In [9]:
train(model, optimizer, criterion, loaders['train'], valid_dataloader=loaders['val'], epochs=2)

Validation epoch 0: 100%|█████████████████████████████████████████████████████████████████████████████████████████████| 1277/1277 [07:44<00:00,  2.75it/s]


epoch=0   train_loss_total=nan  valid_loss_total=nan  valid_acc_total=0.5327


Training epoch 1:  17%|████████████████▍                                                                               | 874/5107 [10:45<52:07,  1.35it/s]

KeyboardInterrupt



In [ ]:
test(model, loaders['test'])